In [1]:
!nvidia-smi

Tue Dec 16 08:42:47 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.95.05              Driver Version: 580.95.05      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA H100                    On  |   00000000:C6:00.0 Off |                    0 |
| N/A   33C    P0             68W /  700W |       0MiB /  95830MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [2]:
import os
import re
import math
from tqdm import tqdm
#from google.colab import userdata
from huggingface_hub import login
import torch
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, set_seed, BitsAndBytesConfig
from datasets import load_dataset, Dataset, DatasetDict
import wandb
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig
from datetime import datetime
import matplotlib.pyplot as plt

from dotenv import load_dotenv
load_dotenv()

/home/yhuang/fine_tuning_project/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/yhuang/fine_tuning_project/.venv/lib/python3.12/site-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/home/yhuang/fine_tuning_project/.venv/lib/python3.12/site-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarni

True

In [3]:
hf_token = os.getenv('HF_TOKEN')
login(hf_token, add_to_git_credential=True)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [4]:
# Constants
BASE_MODEL = "Qwen/Qwen3-8B"
PROJECT_NAME = "Qwen3-8B-ruozhi_v2"
HF_USER = "franzyellow"

# Data
DATASET_NAME = f"{HF_USER}/ruozhiba_punchline_ft"
MAX_SEQUENCE_LENGTH = 150

# Run name for saving the model in the hub
# for better model version management
RUN_NAME =  f"{datetime.now():%Y-%m-%d_%H.%M.%S}"
PROJECT_RUN_NAME = f"{PROJECT_NAME}-{RUN_NAME}"
HUB_MODEL_NAME = f"{HF_USER}/{PROJECT_RUN_NAME}"

# Hyperparameters for QLoRA
LORA_R = 32 # can downgrade to 8 when resource is limited
LORA_ALPHA = 64 # 2r
TARGET_MODULES = ["q_proj", "v_proj", "k_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
LORA_DROPOUT = 0.1
QUANT_4_BIT = False

# Hyperparameters for Training

EPOCHS = 5
BATCH_SIZE = 16 # on an A100 box this can go up to 16
GRADIENT_ACCUMULATION_STEPS = 1 # not really applying here
LEARNING_RATE = 2e-5
LR_SCHEDULER_TYPE = 'cosine' # dynamically lowering the LR in later epoch, cosine is a good shape for the purpose
WARMUP_RATIO = 0.03 # lowering the learning rate at the early steps and warming it up later where cosine scheduler becomes more important
OPTIMIZER = "paged_adamw_32bit" # https://huggingface.co/docs/transformers/main/en/perf_train_gpu_one#optimizer-choice

# Admin config - note that SAVE_STEPS is how often it will upload to the hub
# I've changed this from 5000 to 2000 so that you get more frequent saves

STEPS = 50 # WANDB update freq
SAVE_STEPS = 2000 # model saving freq
LOG_TO_WANDB = True

%matplotlib inline

In [5]:
# Log in to Weights & Biases
wandb_api_key = os.getenv('WANDB_API_KEY')
wandb.login()

# Configure Weights & Biases to record against our project
os.environ["WANDB_PROJECT"] = PROJECT_NAME
os.environ["WANDB_LOG_MODEL"] = "checkpoint" if LOG_TO_WANDB else "end"
os.environ["WANDB_WATCH"] = "gradients"

wandb: Currently logged in as: franzhuang027 (franzhuang027-university-of-amsterdam) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


## Loading data

In [6]:
dataset = load_dataset(DATASET_NAME)
train = dataset['train']
train[0]

Generating train split: 100%|██████████| 3439/3439 [00:00<00:00, 505615.94 examples/s]


{'instruction': '和尚去参加漫展了',
 'output': '他化二次缘了',
 'text': '和尚去参加漫展了ANSWER:他化二次缘了'}

In [7]:
train[10]

{'instruction': '最近有什么新的警方查案的新闻吗？',
 'output': '警方在一家雇用童工的火力发电厂里查出大批火力少年王。',
 'text': '最近有什么新的警方查案的新闻吗？ANSWER:警方在一家雇用童工的火力发电厂里查出大批火力少年王。'}

In [8]:
if LOG_TO_WANDB:
  wandb.init(project=PROJECT_NAME, name=RUN_NAME)

## Loading the model

In [9]:
# pick the right quantization

if QUANT_4_BIT:
  quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4"
  )
else:
  quant_config = BitsAndBytesConfig(
    load_in_8bit=True,
    bnb_8bit_compute_dtype=torch.bfloat16
  )

In [10]:
# Load the Tokenizer and the Model

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quant_config,
    device_map="auto",
)
base_model.generation_config.pad_token_id = tokenizer.pad_token_id

print(f"Memory footprint: {base_model.get_memory_footprint() / 1e6:.1f} MB")

Loading checkpoint shards: 100%|██████████| 5/5 [00:07<00:00,  1.54s/it]


Memory footprint: 9435.7 MB


## Data Collator

In [11]:
from trl import DataCollatorForCompletionOnlyLM
response_template = "ANSWER:" # what is the chunk of text that indicates the prediction target
collator = DataCollatorForCompletionOnlyLM(response_template, tokenizer=tokenizer) # building the intended mask behind the scene

## Training Config

In [12]:
# First, specify the configuration parameters for LoRA

lora_parameters = LoraConfig(
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    r=LORA_R,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=TARGET_MODULES,
)

# Next, specify the general configuration parameters for training

train_parameters = SFTConfig(
    output_dir=PROJECT_RUN_NAME,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=1,
    eval_strategy="no", # if 'yes', test performance on the held-out validation set repeatedly
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    optim=OPTIMIZER,
    save_steps=SAVE_STEPS,
    save_total_limit=10,
    logging_steps=STEPS,
    learning_rate=LEARNING_RATE,
    weight_decay=0.001,
    fp16=False,
    bf16=True,
    max_grad_norm=0.3,
    max_steps=-1,
    warmup_ratio=WARMUP_RATIO,
    group_by_length=True,
    lr_scheduler_type=LR_SCHEDULER_TYPE,
    report_to="wandb" if LOG_TO_WANDB else None,
    run_name=RUN_NAME,
    max_seq_length=MAX_SEQUENCE_LENGTH,
    dataset_text_field="text",
    save_strategy="steps",
    hub_strategy="every_save",
    push_to_hub=True,
    hub_model_id=HUB_MODEL_NAME,
    hub_private_repo=True
)

# And now, the Supervised Fine Tuning Trainer will carry out the fine-tuning
# Given these 2 sets of configuration parameters
# The latest version of trl is showing a warning about labels - please ignore this warning
# But let me know if you don't see good training results (loss coming down).

fine_tuning = SFTTrainer(
    model=base_model,
    train_dataset=train,
    peft_config=lora_parameters,
    args=train_parameters,
    data_collator=collator
  )

Map: 100%|██████████| 3439/3439 [00:00<00:00, 47937.85 examples/s]


## Implementation

In [13]:
# Fine-tune!
fine_tuning.train()

# Push our fine-tuned model to Hugging Face
fine_tuning.model.push_to_hub(PROJECT_RUN_NAME, private=True)
print(f"Saved to the hub: {PROJECT_RUN_NAME}")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


/home/yhuang/fine_tuning_project/.venv/lib/python3.12/site-packages/bitsandbytes/autograd/_functions.py:181: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")
/home/yhuang/fine_tuning_project/.venv/lib/python3.12/site-packages/bitsandbytes/autograd/_functions.py:181: UserWarning: MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


Step,Training Loss
50,3.860200
100,3.319000
150,3.286700
200,3.181500
250,2.986200
300,2.966100
350,2.987200
400,2.925400
450,2.839700
500,2.634700


/home/yhuang/fine_tuning_project/.venv/lib/python3.12/site-packages/trl/trainer/utils.py:139: UserWarning: Could not find response key `ANSWER:` in the following instance: 今天有什么新闻值得关注吗？ANSWER:CCTV-38今日焦点关注为什么八只小崽子为了一朵小红花打架住院。. This instance will be ignored in loss calculation. Note, if this happens often, consider increasing the `max_seq_length`.
  warnings.warn(
/home/yhuang/fine_tuning_project/.venv/lib/python3.12/site-packages/trl/trainer/utils.py:139: UserWarning: Could not find response key `ANSWER:` in the following instance: 今天有什么新闻值得关注吗？ANSWER:CCTV-38今日焦点关注为什么八只小崽子为了一朵小红花打架住院。<|im_end|>. This instance will be ignored in loss calculation. Note, if this happens often, consider increasing the `max_seq_length`.
  warnings.warn(
wandb: Adding directory to artifact (Qwen3-8B-ruozhi_v2-2025-12-16_08.43.01/checkpoint-1075)... Done. 3.0s
Processing Files (1 / 1): 100%|██████████|  349MB /  349MB,  182MB/s  
New Data Upload: |          |  0.00B /  0.00B,  0.00B/s  
No files have been mod

Saved to the hub: Qwen3-8B-ruozhi_v2-2025-12-16_08.43.01


In [14]:
if LOG_TO_WANDB:
  wandb.finish()

wandb: ERROR The nbformat package was not found. It is required to save notebook history.


train/epoch,▁▁▂▂▂▃▃▃▄▄▄▅▅▅▆▆▆▇▇▇██
train/global_step,▁▁▂▂▂▃▃▃▄▄▄▅▅▅▆▆▆▇▇▇██
train/grad_norm,▂▂▂▃▁▂▂▂▂▃▃▃▃▄▄▄▆▇▇▇█
train/learning_rate,████▇▇▇▆▆▅▅▄▄▃▃▂▂▁▁▁▁
train/loss,█▆▆▅▅▅▅▄▄▃▃▃▃▂▂▂▂▁▁▁▁
total_flos,2.2000367263494144e+16
train/epoch,5
train/global_step,1075
train/grad_norm,9.48722
train/learning_rate,0.0
train/loss,2.1111
